# 1.3

Моделирование выборок Пуассоновского распределения


In [ ]:
import math
from random import random

def poisson_one(theta: float) -> int:
    y = random()
    k = 0
    p = math.exp(-theta)
    F = p
    while y > F:
        k += 1
        p *= theta / k
        F += p
    return k

def poisson_sample(theta: float, n: int) -> list:
    rez = []
    for _ in range(n):
        val = poisson_one(theta)
        rez.append(val)
    return rez

teta = 21.0
n = 100
sample = poisson_sample(teta, n)
print(sample)


# 2.1
Для каждой из выбранных случайных величин необходимо построить по 5
выборок следующих объемов n = {5, 10, 100, 200, 400, 600, 800, 1000}.


Пуассоновское распределение



In [ ]:
size = [5, 10, 100, 200, 400, 600, 800, 1000]

samples_storage = {}

for i in size:
    print(f" 5 выборок объема n = {i} ")
    five_samples = []
    for k in range(5):
      sample = poisson_sample(teta, i)
      five_samples.append(sample)
      print(sample)
    samples_storage[i] = five_samples
    print("")

# 2.2

In [ ]:
samples_storage[5]

Функция ЭФР

In [ ]:
# F_n(x) = (количество элементов <= x) / n
def get_ecdf_value(sample: list, x: float) -> float:
    count = 0
    for val in sample:
        if val <= x:
            count += 1
    return count / len(sample)

Общий график ЭФР и теоретической функции


In [ ]:
import matplotlib.pyplot as plt
import math

# --- Ваши исходные функции и параметры (предполагаем, что они определены) ---
# teta = ...
# size = [...]
# samples_storage = {...}
# def get_ecdf_value(sample, x): ...

def theoretical_cdf(theta: float, x: float) -> float:
    if x < 0: return 0.0
    k_max = int(x)
    cdf = 0.0
    p = math.exp(-theta)
    cdf += p
    for k in range(1, k_max + 1):
        p *= theta / k
        cdf += p
    return min(cdf, 1.0)

# --- Настройка отрисовки ---
x_points = list(range(0, 45))

# Создаем "фигуру" и массив "осей" (axes).
# ncols=1 означает, что графики будут идти столбиком (один под другим).
# figsize делаем высоким, чтобы графики не сплющивались.
num_plots = len(size)
fig, axes = plt.subplots(nrows=num_plots, ncols=1, figsize=(10, 5 * num_plots))

# Если размер выборки всего один, matplotlib вернет не массив, а один объект axes.
# Превращаем его в список, чтобы цикл работал универсально.
if num_plots == 1:
    axes = [axes]

# Рассчитаем теорию один раз, она одинакова для всех графиков
y_theor = [theoretical_cdf(teta, x) for x in x_points]

# Проходимся одновременно по осям (графикам) и размерам выборок
for ax, n in zip(axes, size):
    five_sample = samples_storage[n]

    # 1. Рисуем 5 эмпирических выборок на текущем графике (ax)
    for i, sample in enumerate(five_sample):
        y_values = [get_ecdf_value(sample, x) for x in x_points]
        # Рисуем тонкие линии с прозрачностью, чтобы было видно наложения
        ax.step(x_points, y_values, where='post', alpha=0.5, label=f'Выборка {i+1}')

    # 2. Рисуем теоретическую кривую поверх выборок
    ax.step(x_points, y_theor, where='post', color='black', linewidth=2, linestyle='--', label='Теория')

    # Настройки для текущего графика
    ax.set_title(f'Эмпирические функции (n={n}) vs Теория')
    ax.set_ylabel('F(x)')
    ax.grid(True, alpha=0.3)

    # Легенду можно добавить только на первый график или на все (по желанию)
    ax.legend(loc='lower right')

# Подпись оси X только для нижнего графика, чтобы не захламлять
axes[-1].set_xlabel('x')

plt.tight_layout() # Автоматически выравнивает отступы между графиками
plt.show()


In [ ]:
for i in range(1, 1):
  print(i)

Вычисление статистики D_mn


In [ ]:
def calculate_D_mn(sample1: list, sample2: list) -> float:
    n = len(sample1)
    m = len(sample2)

    # находим все точки скачков (объединяем уникальные значения двух выборок)
    all_jumps = sorted(list(set(sample1 + sample2)))

    # ищем максимальное расхождение (supremum)
    max_diff = 0.0
    for x in all_jumps:
        f_n = get_ecdf_value(sample1, x)
        f_m = get_ecdf_value(sample2, x)
        diff = abs(f_n - f_m)
        if diff > max_diff:
            max_diff = diff

    # считаем итоговую статистику по формуле
    D = math.sqrt((n * m) / (n + m)) * max_diff
    return D

print(f"\nТаблица значений D_mn (theta={teta}):")
print("-" * 75)

# шапка
header = f"{'n \\ m':<6} | " + " | ".join([f"{s:<6}" for s in size])
print(header)
print("-" * len(header))

# тело
for n_row in size:
    row_cells = [f"{n_row:<6}"]
    for n_col in size:
        s1 = samples_storage[n_row][0]
        s2 = samples_storage[n_col][0]

        val = calculate_D_mn(s1, s2)
        row_cells.append(f"{val:.4f}")
    print(" | ".join(row_cells))


#2.3 Построение гистограммы и полигона частот

In [ ]:
import matplotlib.pyplot as plt
import math

# Функция вероятности Пуассона P(X=k) для теоретических точек
def poisson_pmf(theta: float, k: int) -> float:
    # P(k) = (theta^k / k!) * e^(-theta)
    return (theta**k * math.exp(-theta)) / math.factorial(k)

# Диапазон значений k, которые будем отображать на графике (от 5 до 40)
# (там лежит основная масса вероятности для theta=21)
x_range = list(range(5, 41))

# Считаем теоретические вероятности один раз (идеальная "горка")
y_theor = [poisson_pmf(teta, k) for k in x_range]

# Строим графики для каждого объема выборки из нашего хранилища
plt.figure(figsize=(15, 20)) # Делаем большой рисунок, чтобы все влезло

for i, n in enumerate(size):
    plt.subplot(4, 2, i + 1) # Сетка 4 строки, 2 столбца

    sample = samples_storage[n][0] # Берем сохраненную выборку

    # 1. Строим Гистограмму (столбики)
    # bins - границы столбиков. Выравниваем их по целым числам.
    # density=True - нормируем, чтобы сумма площадей была = 1
    bins = range(min(sample), max(sample) + 2)
    counts, _, _ = plt.hist(sample, bins=bins, density=True,
                            alpha=0.4, color='skyblue', edgecolor='black', label='Гистограмма')

    # 2. Строим Полигон частот (синяя линия)
    # Полигон соединяет середины верхушек столбиков.
    # Сдвигаем координаты X на 0.5 вправо, чтобы попасть в центр столбика.
    bin_centers = [b + 0.5 for b in bins[:-1]]
    plt.plot(bin_centers, counts, 'b-o', linewidth=2, label='Полигон частот')

    # 3. Строим Теоретические точки (красная пунктирная линия)
    plt.plot(x_range, y_theor, 'r--x', linewidth=2, label='Теория P(k)')

    plt.title(f'Объем выборки n = {n}')
    plt.xlabel('Значение k')
    plt.ylabel('Вероятность')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout() # Чтобы графики не наезжали друг на друга
plt.show()

# 2.4 Вычисление выборочных моментов

In [ ]:
#  Вычисляет выборочное среднее: sum(x_i) / n
def calculate_mean(sample: list) -> float:
    if not sample: return 0.0
    return sum(sample) / len(sample)
#   Вычисляет выборочную дисперсию: sum((x_i - mean)^2) / n
def calculate_variance(sample: list) -> float:
    n = len(sample)
    if n <= 1: return 0.0

    mean_val = calculate_mean(sample)
    sum_sq_diff = 0.0
    for x in sample:
        sum_sq_diff += (x - mean_val) ** 2

    return sum_sq_diff / n

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА ВЫБОРОЧНЫХ МОМЕНТОВ (Истинное значение мат.ожидания и дисперсии: theta = {teta})")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} |{'Выб. среднее':<15} | {'Выб. дисперсия':<15} | {'|Mean - theta|':<15} | {'|Var - theta|':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in size:
  for sample_number in range(5):
    sample = samples_storage[n][sample_number]
    mean_val = calculate_mean(sample)
    var_val = calculate_variance(sample)
    diff_mean = abs(mean_val - teta)
    diff_var = abs(var_val - teta)

    # Выводим строку
    print(f"{n:<10} | {sample_number+1:<15} |{mean_val:<15.4f} | {var_val:<15.4f} | {diff_mean:<15.4f} | {diff_var:<15.4f}")
  print("-" * len(header))
print("-" * len(header))

1. Выборочное среднее:
   - Является несмещенной, состоятельной и эффективной оценкой параметра theta.
   - Мы видим, что с ростом n значение всё ближе к 21.0.

2. Выборочная дисперсия:
   - Является состоятельной оценкой дисперсии.
   - Так как для Пуассона D(X) = M(X) = theta, выборочная дисперсия тоже стремится к 21.0
   - Данная оценка является смещенной. Несмещенная оценка (исправленная дисперсия) требовала бы деления на n-1
   

# 3.1

In [ ]:
def calc_omm(sample: list):
  return 1.0*sum(sample)/len(sample)

def calc_omp(sample: list):
  return 1.0*sum(sample)/len(sample)

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА ОЦЕНОК ПАРАМЕТРА theta (Истинное значение: theta = {teta})")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} |{'Оценка ММ':<15} | {'Оценка МП':<15} | {'Точность ОММ':<15} | {'Точность ОМП':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in size:
  for sample_number in range(5):
    sample = samples_storage[n][sample_number]
    omm = calc_omm(sample)
    omp = calc_omp(sample)
    # Выводим строку
    print(f"{n:<10} | {sample_number+1:<15} | {omm:<15.4f} | {omp:<15.4f} | {abs(omm - teta):<15.4f} | {abs(omp - teta):<15.4f}")
  print("-" * len(header))
print("-" * len(header))

#3.2

In [ ]:
def opt_teta(sample):
  return sum(sample)/len(sample)

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА ОПТИМАЛЬНЫХ ОЦЕНОК ПАРАМЕТРА theta (Истинное значение: theta = {teta})")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} |{'Оценка':<15} | {'Точность ОМП':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in size:
  for sample_number in range(5):
    sample = samples_storage[n][sample_number]
    opt = opt_teta(sample)
    # Выводим строку
    print(f"{n:<10} | {sample_number+1:<15} | {opt:<15.4f} {abs(opt - teta):<15.4f}")
  print("-" * len(header))
print("-" * len(header))

#3.3

In [ ]:
import kagglehub
path = kagglehub.dataset_download("azizozmen/police-killings-us")

In [ ]:
import os
import pandas as pd

df = pd.read_csv(os.path.join(path, "PoliceKillingsUS.csv"), encoding="windows-1252")
print(df.head())

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%y', errors='coerce').dt.normalize()

In [ ]:
weekly = (
    df.groupby(pd.Grouper(key='date', freq='W-MON'))
      .size()
      .reset_index(name='n_rows')
      .sort_values('date')
)


In [ ]:
weekly.head(10)

In [ ]:
ax = weekly['n_rows'].plot(kind='hist', bins=15)
ax.set_xlabel('Число строк в неделе')
ax.set_ylabel('Частота недель')
ax.set_title('Распределение чисел строк по неделям')
plt.show()

In [ ]:
sample_list = weekly['n_rows'].tolist()

In [ ]:
opt_real = opt_teta(sample_list)
mean_real = calculate_mean(sample_list)
var_real = calculate_variance(sample_list)

print("ЗНАЧЕНИЯ ВЫБОРОЧНЫХ МОМЕНТОВ И ОПТИМАЛЬНОЙ ОЦЕНКИ ДЛЯ РЕАЛЬНЫХ ДАННЫХ")
print(f"Выборочное матожидание: {mean_real:.3f}")
print(f"Выборочная дисперсия: {var_real:.3f}")
print(f"Оптимальная оценка theta: {opt_real:.3f}")


#4.1

## Критерий хи-квадрат




In [ ]:
import numpy as np
from scipy.stats import chi2

alpha = 0.05

In [ ]:
import numpy as np
from scipy.stats import poisson, chi2

def build_v_p(sample, k, teta):
    sample = np.asarray(sample)
    n = len(sample)
    m = int(sample.max())

    domain = np.arange(0, m + 1)
    if k > len(domain):
        k = len(domain)

    groups = np.array_split(domain, k)

    v = []
    p = []

    for i, group in enumerate(groups):
        if len(group) == 0: continue

        L = int(group[0])
        R = int(group[-1])

        if i == k - 1:
            v_i = np.sum(sample >= L)
            p_i = 1.0 - theoretical_cdf(teta, L-1)
        else:
            v_i = np.sum((sample >= L) & (sample <= R))
            p_i = theoretical_cdf(teta, R) - theoretical_cdf(teta, L-1)

        v.append(v_i)
        p.append(p_i)

    return np.array(v), np.array(p), k


In [ ]:
def calculate_Hi2(v, p):
  N = len(v)
  n = sum(v)

  hi2 = 0.0
  for j in range(N):
    hi2 += (v[j] - n * p[j])**2 / (n * p[j])
  return hi2

In [ ]:
def criteriyHi2(sample, k, teta=teta, r=0, alpha=alpha):
  v, p, k_real = build_v_p(sample, k, teta)
  hi2_n = calculate_Hi2(v, p)
  t = chi2.ppf(1 - alpha, k_real - 1 - r)
  return hi2_n >= t, hi2_n, t, k_real

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА КРИТЕРИЯ ХИ КВАДРАТ ДЛЯ ПРОСТОЙ ГИПОТЕЗЫ")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} | {'Кол. интервалов':<15} |{'Вывод':<15} | {'Х^2_n(X)':<15} | {'T_alpha':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in size:
  for count_interval in [3, 5, 10, 30]:
    for sample_number in range(5):
      sample = samples_storage[n][sample_number]
      ans, T, t, k_real = criteriyHi2(sample, count_interval)
      if ans:
        ans_str = "H_0 отвергается"
      else:
        ans_str = "H_0 принимается"
      print(f"{n:<10} | {sample_number+1:<15}| {k_real:<15} |{ans_str:<15} | {T:<15.7f} | {t:<15.4f}")
    print("-" * len(header))
  print("-" * len(header))

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА КРИТЕРИЯ ХИ КВАДРАТ ДЛЯ СЛОЖНОЙ ГИПОТЕЗЫ")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} | {'Кол. интервалов':<15} |{'Оценка МП':<15} |{'Вывод':<15} | {'Х^2_n(X)':<15} | {'T_alpha':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in size:
  for count_interval in [3, 5, 10, 30]:
    for sample_number in range(5):
      sample = samples_storage[n][sample_number]
      tet = calc_omp(sample)
      ans, T, t, k_real = criteriyHi2(sample, count_interval, tet, 1)
      if ans:
        ans_str = "H_0 отвергается"
      else:
        ans_str = "H_0 принимается"
      print(f"{n:<10} | {sample_number+1:<15} | {k_real:<12} | {tet:<15.7f} | {ans_str:<15} | {T:<15.7f} | {t:<15.4f}")
    print("-" * len(header))
  print("-" * len(header))